# BTCUSDC Order-Flow Study: Persistent Imbalance Visual Diagnostics

This notebook is for eyeballing whether strong trade-sign imbalance behaves differently when it persists. It does not test returns yet. It marks windows where last-`N` sign imbalance is high, then optionally marks the subset where the same-side imbalance persisted for a user-defined event or clock duration.


## Load Data

Use the shared order-flow loader and trade-frame alignment helper. The analysis below starts from trade-aligned rows with timestamp, trade quantity, aggressor sign, and latest known top-of-book midprice.


In [1]:
\
from pathlib import Path
import json
import sys
import uuid

import numpy as np
import pandas as pd
import plotly.graph_objects as go
from plotly.subplots import make_subplots
import ipywidgets as widgets
from IPython.display import HTML, display


def find_backtester_root() -> Path:
    for candidate in [Path.cwd().resolve(), *Path.cwd().resolve().parents]:
        if (candidate / "stats").is_dir() and (candidate / "notebooks").is_dir():
            return candidate
    raise FileNotFoundError("Could not locate the exchange-data-backtester project root")


PROJECT_ROOT = find_backtester_root()
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

from stats.features import make_trade_frame
from stats.notebook import load_orderflow_day

pd.set_option("display.max_rows", 200)
pd.set_option("display.max_columns", 200)

EXCHANGE = "binance"
SYMBOL = "BTCUSDC"
REFERENCE_DAY = "20260223"
REPLAY_ON_GAP = "skip-segment"

_, trades, top, day_summary = load_orderflow_day(
    day=REFERENCE_DAY,
    symbol=SYMBOL,
    exchange=EXCHANGE,
    replay_on_gap=REPLAY_ON_GAP,
)

trade_frame = make_trade_frame(trades, top)

display(day_summary.to_frame("value"))
summary = pd.Series({
    "trade_frame_rows": len(trade_frame),
    "start_utc": trade_frame["ts"].min(),
    "end_utc": trade_frame["ts"].max(),
    "buy_trades": int((trade_frame["aggr_sign"] > 0).sum()),
    "sell_trades": int((trade_frame["aggr_sign"] < 0).sum()),
})
display(summary.to_frame("value"))
display(trade_frame.head())


,value
exchange,binance
symbol,BTCUSDC
day,20260223
day_dir,/Users/hoangdeveloper/PycharmProjects/exchange...
trades_rows,868008
top_rows,722893
trade_start_utc,2026-02-23 01:00:04.958000+00:00
trade_end_utc,2026-02-23 23:15:00.320000+00:00
top_start_utc,2026-02-23 01:00:04.735000+00:00
top_end_utc,2026-02-23 23:14:59.835000+00:00


,value
trade_frame_rows,868008
start_utc,2026-02-23 01:00:04.958000+00:00
end_utc,2026-02-23 23:15:00.320000+00:00
buy_trades,445726
sell_trades,422282


,ts,price,qty,aggr_sign,mid_at_book
0,2026-02-23 01:00:04.958000+00:00,66704.44,0.00009,1.0,66704.435
1,2026-02-23 01:00:04.958000+00:00,66704.44,0.00017,1.0,66704.435
2,2026-02-23 01:00:05.037000+00:00,66704.44,0.00989,1.0,66704.435
3,2026-02-23 01:00:05.087000+00:00,66704.43,0.00123,-1.0,66704.435
4,2026-02-23 01:00:05.087000+00:00,66702.01,0.00008,-1.0,66704.435


## Persistent Imbalance Plot

Base imbalance:

`signed_share_N = sum(last N trade signs) / N`

A trade sign is `+1` for buyer-initiated trades and `-1` for seller-initiated trades. A buy imbalance is active when `signed_share_N >= imb th`; a sell imbalance is active when `signed_share_N <= -imb th`.

Editable controls:

- `N trades`: number of most recent trades used to compute `signed_share_N`, rolling volume, and trades/sec.
- `imb th`: absolute sign-imbalance cutoff. Example: `0.70` means at least 70% of the last `N` trades were on one side.
- `persist basis`: how persistence is measured.
  - `event windows`: require the same-side high-imbalance condition for a minimum number of consecutive computed imbalance rows.
  - `clock time`: require the same-side high-imbalance condition to last for a minimum wall-clock duration.
  - `clock rule`: for clock time, choose either `run-based` or `mean-based`.
- `persist windows`: minimum number of consecutive high-imbalance rows for event-window persistence.
- `persist min`: minimum wall-clock duration for clock-time persistence.
- `show all imbalances`: show every high-imbalance point on the price panel.
- `show persistent`: show persistent high-imbalance points on the price panel.
- Volume imbalance uses the same threshold logic on buy volume versus sell volume. Count imbalance markers stay circular; volume imbalance markers are square.

Precise persistence definition:

- First classify each row as `+1` buy-active, `-1` sell-active, or `0` inactive using `imb th`.
- Consecutive rows with the same nonzero active sign form a same-side imbalance run.
- Event persistence is true once the row position inside that run is at least `persist windows`.
- Clock run-based persistence is true once elapsed time from the start of that run is at least `persist min`.
- Clock mean-based persistence is true once the rolling mean of `signed_share_N` over the last `persist min` stays beyond the same sign threshold.

Because these are rolling last-`N` windows, adjacent rows overlap heavily. This plot is a visual diagnostic, not an independent-event count.


In [2]:
DEFAULT_EVENT_N = 100
DEFAULT_IMBALANCE_THRESHOLD = 0.70
MAX_PLOT_ROWS = 50_000


def _sample_evenly(frame: pd.DataFrame, *, max_rows: int) -> pd.DataFrame:
    if len(frame) <= max_rows:
        return frame.copy()
    step = int(np.ceil(len(frame) / max_rows))
    return frame.iloc[::step].copy()


def build_event_features(trade_frame: pd.DataFrame, *, event_n: int) -> pd.DataFrame:
    event_n = int(event_n)
    if event_n <= 1:
        raise ValueError("event_n must be greater than 1")
    if len(trade_frame) < event_n:
        raise ValueError(f"not enough trades for event_n={event_n}")

    out = trade_frame[["ts", "mid_at_book", "qty", "aggr_sign"]].copy().sort_values("ts").reset_index(drop=True)
    signs = out["aggr_sign"].to_numpy(dtype=float)
    qty = out["qty"].to_numpy(dtype=float)
    buy_qty = np.where(signs > 0, qty, 0.0)
    sell_qty = np.where(signs < 0, qty, 0.0)
    out["net_count_N"] = pd.Series(signs).rolling(event_n, min_periods=event_n).sum().to_numpy()
    out["signed_share_N"] = out["net_count_N"] / float(event_n)
    out["total_volume_N"] = pd.Series(qty).rolling(event_n, min_periods=event_n).sum().to_numpy()
    out["buy_volume_N"] = pd.Series(buy_qty).rolling(event_n, min_periods=event_n).sum().to_numpy()
    out["sell_volume_N"] = pd.Series(sell_qty).rolling(event_n, min_periods=event_n).sum().to_numpy()
    out["volume_share_N"] = (out["buy_volume_N"] - out["sell_volume_N"]) / out["total_volume_N"].replace(0.0, np.nan)
    event_span_sec = (out["ts"] - out["ts"].shift(event_n - 1)).dt.total_seconds().replace(0.0, np.nan)
    out["trades_per_sec_N"] = float(event_n) / event_span_sec
    return out.dropna(subset=["mid_at_book", "signed_share_N", "total_volume_N", "buy_volume_N", "sell_volume_N", "volume_share_N", "trades_per_sec_N"]).reset_index(drop=True)


def add_persistence_flags(
    features: pd.DataFrame,
    *,
    threshold: float,
    persist_basis: str,
    clock_rule: str,
    persist_windows: int,
    persist_minutes: float,
) -> pd.DataFrame:
    out = features.copy()
    high_sign = np.select(
        [out["signed_share_N"] >= threshold, out["signed_share_N"] <= -threshold],
        [1, -1],
        default=0,
    )
    out["imbalance_side"] = high_sign.astype(int)
    side = pd.Series(out["imbalance_side"].to_numpy(), index=out.index)
    run_id = side.ne(side.shift()).cumsum()
    out["imbalance_run_id"] = run_id.where(side.ne(0), np.nan)
    out["run_pos"] = out.groupby("imbalance_run_id", dropna=True).cumcount() + 1
    run_start_ts = out.groupby("imbalance_run_id", dropna=True)["ts"].transform("first")
    out["run_elapsed_sec"] = (out["ts"] - run_start_ts).dt.total_seconds()
    clock_mean_share = (
        out.set_index("ts")["signed_share_N"]
        .rolling(pd.Timedelta(minutes=float(persist_minutes)), min_periods=1)
        .mean()
        .to_numpy()
    )
    out["clock_mean_signed_share"] = clock_mean_share
    volume_side = np.select(
        [out["volume_share_N"] >= threshold, out["volume_share_N"] <= -threshold],
        [1, -1],
        default=0,
    )
    out["volume_imbalance_side"] = volume_side.astype(int)
    volume_side_series = pd.Series(out["volume_imbalance_side"].to_numpy(), index=out.index)
    volume_run_id = volume_side_series.ne(volume_side_series.shift()).cumsum()
    out["volume_imbalance_run_id"] = volume_run_id.where(volume_side_series.ne(0), np.nan)
    out["volume_run_pos"] = out.groupby("volume_imbalance_run_id", dropna=True).cumcount() + 1
    volume_run_start_ts = out.groupby("volume_imbalance_run_id", dropna=True)["ts"].transform("first")
    out["volume_run_elapsed_sec"] = (out["ts"] - volume_run_start_ts).dt.total_seconds()
    volume_clock_mean_share = (
        out.set_index("ts")["volume_share_N"]
        .rolling(pd.Timedelta(minutes=float(persist_minutes)), min_periods=1)
        .mean()
        .to_numpy()
    )
    out["clock_mean_volume_share"] = volume_clock_mean_share

    event_mask = (out["imbalance_side"] != 0) & (out["run_pos"] >= int(persist_windows))
    clock_mask = (out["imbalance_side"] != 0) & (out["run_elapsed_sec"] >= float(persist_minutes) * 60.0)
    clock_mean_mask = (out["imbalance_side"] != 0) & (
        (out["clock_mean_signed_share"] >= threshold) | (out["clock_mean_signed_share"] <= -threshold)
    )
    volume_event_mask = (out["volume_imbalance_side"] != 0) & (out["volume_run_pos"] >= int(persist_windows))
    volume_clock_mask = (out["volume_imbalance_side"] != 0) & (out["volume_run_elapsed_sec"] >= float(persist_minutes) * 60.0)
    volume_clock_mean_mask = (out["volume_imbalance_side"] != 0) & (
        (out["clock_mean_volume_share"] >= threshold) | (out["clock_mean_volume_share"] <= -threshold)
    )
    if persist_basis == "clock time":
        out["persistent_imbalance"] = clock_mean_mask if clock_rule == "mean-based" else clock_mask
        out["persistent_volume_imbalance"] = volume_clock_mean_mask if clock_rule == "mean-based" else volume_clock_mask
    else:
        out["persistent_imbalance"] = event_mask
        out["persistent_volume_imbalance"] = volume_event_mask
    out["persistent_side"] = np.where(out["persistent_imbalance"], out["imbalance_side"], 0).astype(int)
    out["persistent_volume_side"] = np.where(out["persistent_volume_imbalance"], out["volume_imbalance_side"], 0).astype(int)
    return out


def make_persistence_figure(
    features: pd.DataFrame,
    *,
    threshold: float,
    show_all_imbalances: bool,
    show_persistent: bool,
    title: str,
) -> go.Figure:
    plot_df = _sample_evenly(features, max_rows=MAX_PLOT_ROWS)
    price_df = features[["ts", "mid_at_book"]].copy()
    all_buy = features.loc[features["imbalance_side"] > 0]
    all_sell = features.loc[features["imbalance_side"] < 0]
    persistent_buy = features.loc[features["persistent_side"] > 0]
    persistent_sell = features.loc[features["persistent_side"] < 0]
    all_buy_volume = features.loc[features["volume_imbalance_side"] > 0]
    all_sell_volume = features.loc[features["volume_imbalance_side"] < 0]
    persistent_buy_volume = features.loc[features["persistent_volume_side"] > 0]
    persistent_sell_volume = features.loc[features["persistent_volume_side"] < 0]

    fig = make_subplots(
        rows=4,
        cols=1,
        shared_xaxes=True,
        row_heights=[0.45, 0.20, 0.18, 0.17],
        vertical_spacing=0.07,
        subplot_titles=("midprice", "last-N sign imbalance", "last-N total volume", "last-N trades per second"),
    )
    fig.add_trace(go.Scattergl(x=price_df["ts"], y=price_df["mid_at_book"], mode="lines", line=dict(color="#111827", width=1), name="midprice", hoverinfo="skip"), row=1, col=1)

    if show_all_imbalances:
        for data, name, color, symbol in [
            (all_buy, "all buy imbalance", "#86efac", "circle"),
            (all_sell, "all sell imbalance", "#fca5a5", "circle"),
            (all_buy_volume, "all buy volume imbalance", "#60a5fa", "square"),
            (all_sell_volume, "all sell volume imbalance", "#f59e0b", "square"),
        ]:
            fig.add_trace(go.Scattergl(x=data["ts"], y=data["mid_at_book"], mode="markers", marker=dict(color=color, size=5, opacity=0.45, symbol=symbol), name=name), row=1, col=1)

    if show_persistent:
        for data, name, color, symbol in [
            (persistent_buy, "persistent buy imbalance", "#16a34a", "diamond"),
            (persistent_sell, "persistent sell imbalance", "#dc2626", "diamond"),
            (persistent_buy_volume, "persistent buy volume imbalance", "#2563eb", "square-open"),
            (persistent_sell_volume, "persistent sell volume imbalance", "#d97706", "square-open"),
        ]:
            fig.add_trace(go.Scattergl(x=data["ts"], y=data["mid_at_book"], mode="markers", marker=dict(color=color, size=8, opacity=0.95, symbol=symbol), name=name), row=1, col=1)

    fig.add_trace(go.Scattergl(x=plot_df["ts"], y=plot_df["signed_share_N"], mode="lines", line=dict(color="#2563eb", width=1.2), name="signed_share_N"), row=2, col=1)
    fig.add_hline(y=threshold, line_dash="dash", line_color="#16a34a", row=2, col=1)
    fig.add_hline(y=-threshold, line_dash="dash", line_color="#dc2626", row=2, col=1)
    fig.add_trace(go.Scattergl(x=plot_df["ts"], y=plot_df["total_volume_N"], mode="lines", line=dict(color="#7c3aed", width=1.2), name="total_volume_N"), row=3, col=1)
    fig.add_trace(go.Scattergl(x=plot_df["ts"], y=plot_df["trades_per_sec_N"], mode="lines", line=dict(color="#ca8a04", width=1.2), name="trades_per_sec_N"), row=4, col=1)

    hover_x = plot_df["ts"].iloc[0]
    for row in range(1, 5):
        yref = "y domain" if row == 1 else f"y{row} domain"
        fig.add_shape(type="line", xref="x", yref=yref, x0=hover_x, x1=hover_x, y0=0, y1=1, line=dict(color="#6b7280", width=1, dash="dash"), visible=False, layer="above")

    fig.update_layout(height=1100, hovermode="x", legend=dict(orientation="h", yanchor="bottom", y=1.02, xanchor="left", x=0), margin=dict(l=65, r=30, t=90, b=50))
    fig.update_yaxes(title_text="mid", row=1, col=1)
    fig.update_yaxes(title_text="signed share", row=2, col=1)
    fig.update_yaxes(title_text="volume", row=3, col=1)
    fig.update_yaxes(title_text="trades / sec", row=4, col=1)
    fig.update_xaxes(title_text="UTC time", row=4, col=1)
    fig._guide_count = 4
    fig._plot_title = title
    return fig

def render_shared_hover_figure(fig: go.Figure) -> None:
    wrapper_id = f"shared-hover-{uuid.uuid4().hex}"
    guide_count = int(getattr(fig, "_guide_count", 1))
    graph_html = fig.to_html(
        full_html=False,
        include_plotlyjs=True,
        config={"responsive": True, "scrollZoom": True, "displaylogo": False},
    )
    plot_title = getattr(fig, "_plot_title", "")
    html = ('''
<div id="__WRAPPER__" style="margin-bottom: 1rem;">
  <div style="font-size: 18px; font-weight: 600; margin: 0 0 0.5rem 0;">__PLOT_TITLE__</div>
  __GRAPH__
</div>
<script>
(function() {
  const root = document.getElementById(__WRAPPER_JSON__);
  if (!root) {
    return;
  }

  function attach() {
    const graph = root.querySelector('.plotly-graph-div');
    if (!graph || typeof graph.on !== 'function') {
      setTimeout(attach, 50);
      return;
    }

    const guideCount = __GUIDE_COUNT__;
    const shapeStart = Array.isArray(graph.layout.shapes) ? graph.layout.shapes.length - guideCount : -1;
    if (shapeStart < 0) {
      return;
    }
    const shapeIndices = Array.from({ length: guideCount }, (_, idx) => shapeStart + idx);

    function setLine(xValue, visible) {
      const update = {};
      shapeIndices.forEach((shapeIndex) => {
        update[`shapes[${shapeIndex}].visible`] = visible;
        if (xValue !== undefined && xValue !== null) {
          update[`shapes[${shapeIndex}].x0`] = xValue;
          update[`shapes[${shapeIndex}].x1`] = xValue;
        }
      });
      Plotly.relayout(graph, update);
    }

    graph.on('plotly_hover', function(evt) {
      const point = evt && evt.points && evt.points[0];
      if (!point) {
        return;
      }
      setLine(point.x, true);
    });

    graph.on('plotly_unhover', function() {
      setLine(null, false);
    });
  }

  attach();
})();
</script>
'''
    .replace("__WRAPPER__", wrapper_id)
    .replace("__WRAPPER_JSON__", json.dumps(wrapper_id))
    .replace("__GRAPH__", graph_html)
    .replace("__PLOT_TITLE__", plot_title)
    .replace("__GUIDE_COUNT__", str(guide_count))
    )
    display(HTML(html))


def draw_persistence_plot(
    event_n: int,
    threshold: float,
    persist_basis: str,
    clock_rule: str,
    persist_windows: int,
    persist_minutes: float,
    show_all_imbalances: bool,
    show_persistent: bool,
):
    features = build_event_features(trade_frame, event_n=int(event_n))
    features = add_persistence_flags(
        features,
        threshold=float(threshold),
        persist_basis=str(persist_basis),
        clock_rule=str(clock_rule),
        persist_windows=int(persist_windows),
        persist_minutes=float(persist_minutes),
    )
    summary = pd.Series({
        "event_n": int(event_n),
        "imbalance_threshold": float(threshold),
        "persist_basis": persist_basis,
        "clock_rule": clock_rule,
        "persist_windows": int(persist_windows),
        "persist_minutes": float(persist_minutes),
        "rows": len(features),
        "all_buy_imbalance_points": int((features["imbalance_side"] > 0).sum()),
        "all_sell_imbalance_points": int((features["imbalance_side"] < 0).sum()),
        "all_buy_volume_imbalance_points": int((features["volume_imbalance_side"] > 0).sum()),
        "all_sell_volume_imbalance_points": int((features["volume_imbalance_side"] < 0).sum()),
        "persistent_buy_points": int((features["persistent_side"] > 0).sum()),
        "persistent_sell_points": int((features["persistent_side"] < 0).sum()),
        "persistent_buy_volume_points": int((features["persistent_volume_side"] > 0).sum()),
        "persistent_sell_volume_points": int((features["persistent_volume_side"] < 0).sum()),
    })
    display(summary.to_frame("value"))
    fig = make_persistence_figure(
        features,
        threshold=float(threshold),
        show_all_imbalances=bool(show_all_imbalances),
        show_persistent=bool(show_persistent),
        title=f"{SYMBOL} {REFERENCE_DAY}: persistent last-{int(event_n)} trade sign imbalance",
    )
    render_shared_hover_figure(fig)
    global latest_persistence_features, latest_persistence_config
    latest_persistence_features = features
    latest_persistence_config = summary
    return features


N_WIDGET = widgets.IntSlider(value=DEFAULT_EVENT_N, min=20, max=500, step=10, description="N trades", continuous_update=False)
THRESHOLD_WIDGET = widgets.FloatSlider(value=DEFAULT_IMBALANCE_THRESHOLD, min=0.10, max=1.00, step=0.05, description="imb th", continuous_update=False)
PERSIST_BASIS_WIDGET = widgets.Dropdown(options=["event windows", "clock time"], value="event windows", description="persist basis")
PERSIST_CLOCK_RULE_WIDGET = widgets.Dropdown(options=["run-based", "mean-based"], value="run-based", description="clock rule")
PERSIST_WINDOWS_WIDGET = widgets.IntSlider(value=50, min=2, max=500, step=1, description="persist windows", continuous_update=False)
PERSIST_MINUTES_WIDGET = widgets.FloatSlider(value=1.0, min=0.1, max=30.0, step=0.1, description="persist min", continuous_update=False)
SHOW_ALL_WIDGET = widgets.Checkbox(value=True, description="show all imbalances")
SHOW_PERSISTENT_WIDGET = widgets.Checkbox(value=True, description="show persistent")
REFRESH_BUTTON = widgets.Button(description="Refresh plot", button_style="primary")
REFRESH_OUTPUT = widgets.Output()


def _refresh_persistence_plot(_=None):
    with REFRESH_OUTPUT:
        REFRESH_OUTPUT.clear_output(wait=True)
        draw_persistence_plot(
            event_n=N_WIDGET.value,
            threshold=THRESHOLD_WIDGET.value,
            persist_basis=PERSIST_BASIS_WIDGET.value,
            clock_rule=PERSIST_CLOCK_RULE_WIDGET.value,
            persist_windows=PERSIST_WINDOWS_WIDGET.value,
            persist_minutes=PERSIST_MINUTES_WIDGET.value,
            show_all_imbalances=SHOW_ALL_WIDGET.value,
            show_persistent=SHOW_PERSISTENT_WIDGET.value,
        )


def _observe_widget_change(change):
    if change.get("name") == "value":
        _refresh_persistence_plot()


REFRESH_BUTTON.on_click(_refresh_persistence_plot)
for widget in [N_WIDGET, THRESHOLD_WIDGET, PERSIST_BASIS_WIDGET, PERSIST_CLOCK_RULE_WIDGET, PERSIST_WINDOWS_WIDGET, PERSIST_MINUTES_WIDGET, SHOW_ALL_WIDGET, SHOW_PERSISTENT_WIDGET]:
    widget.observe(_observe_widget_change, names="value")

display(widgets.VBox([
    widgets.HBox([N_WIDGET, THRESHOLD_WIDGET]),
    widgets.HBox([PERSIST_BASIS_WIDGET, PERSIST_CLOCK_RULE_WIDGET]),
    widgets.HBox([PERSIST_WINDOWS_WIDGET, PERSIST_MINUTES_WIDGET]),
    widgets.HBox([SHOW_ALL_WIDGET, SHOW_PERSISTENT_WIDGET]),
    REFRESH_BUTTON,
    REFRESH_OUTPUT,
]))
_refresh_persistence_plot()
